## Introduction

The [ZymoBIOMICS mock microbial community](https://github.com/LomanLab/mockcommunity) is a standardized mixture of known bacterial and fungal taxa at defined proportions. Because the true composition is known, it is widely used as a benchmarking and proof-of-concept dataset for evaluating bioinformatics pipelines, feature representations, and ecological analysis methods.

In this homework, you will use the Zymo mock community as a controlled testbed to assess the quality of **GenSLMs sequence embeddings**.



### Background

GenSLMs embeddings are pretrained representations derived from the **GenSLM foundation model** (GitHub: https://github.com/ramanathanlab/genslm).  
They are intended to capture biologically meaningful patterns such as evolutionary or phylogenetic relatedness and shared functional motifs.

There are 10 reference genomes in this dataset. Metagenomic reads have been assembled into contigs. The reference genome IDs correspond to:

- **0**: *Bacillus subtilis*
- **1**: *Cryptococcus neoformans*
- **2**: *Enterococcus faecalis*
- **3**: *Escherichia coli*
- **4**: *Lactobacillus fermentum*
- **5**: *Listeria monocytogenes*
- **6**: *Pseudomonas aeruginosa*
- **7**: *Saccharomyces cerevisiae*
- **8**: *Salmonella enterica*
- **9**: *Staphylococcus aureus*

Among these genomes, **E. coli** and **Salmonella** share ~87.04% similarity, and **Bacillus** and **Staphylococcus** share ~72.25% similarity.



### Hypothesis

We hypothesize that GenSLMs embeddings encode evolutionary and functional signals strongly enough that:

1. A classifier trained on the embeddings can accurately predict contig origin (genome ID).
2. Low-dimensional projections (PCA / UMAP / t-SNE) reveal coherent clusters aligned with genome IDs, including known similarity pairs (e.g., *E. coli* vs. *Salmonella*).
3. Clustering algorithms applied to the embeddings approximate true contig origin groupings with high homogeneity and completeness.



### Strategy

1. Load contig-level embeddings and genome IDs.
2. Perform a stratified train/test split to evaluate generalization.
3. Train multiple multi-class classifiers (Logistic Regression, LightGBM, XGBoost, CatBoost) and select the best model based on macro F1.
4. To speed up classification, reduce the dimensionality using PCA and UMAP, and evaluate how different numbers of components affect classification performance.
5. Visualize the embeddings using PCA, t-SNE, and UMAP with tuned hyperparameters; annotate notable overlaps (e.g., *E. coli* vs. *Salmonella*).
6. Apply clustering (DBSCAN / HDBSCAN / KMeans) on the 2D reduced space and compute homogeneity and completeness scores.




### Evaluation

- **Classification**: macro precision, macro recall, macro F1, accuracy.
- **Visualization**: qualitative cluster separation and overlapping
- **Clustering**: homogeneity_score, completeness_score, number of clusters vs expected genomes, outlier count (if applicable).


### Open-Ended Nature

This assignment is intentionally open-ended — there is no single “correct” setting.  
You are encouraged to explore reasonable hyperparameters.  
For some steps, I also provide my own settings so that everyone starts from the same baseline.


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [4]:
genslm_npz = np.load("zymo_genslm_embeddings.npz")
genslm_npz['e'].shape
genslm_npz['id'].shape

(11724,)

In [5]:
genslm_embed = pd.DataFrame(genslm_npz['e'])
genslm_embed['rid'] = genslm_npz['id']
genslm_embed=genslm_embed.sample(frac=1)
genslm_embed.describe()

,0,1,2,3,4,5,6,7,8,9,...,3831,3832,3833,3834,3835,3836,3837,3838,3839,rid
count,11724.000000,11724.000000,11724.000000,11724.000000,11724.000000,11724.000000,11724.000000,11724.000000,11724.000000,11724.000000,...,11724.000000,11724.000000,11724.000000,11724.000000,11724.000000,11724.000000,11724.000000,11724.000000,11724.000000,11724.000000
mean,-0.172946,0.229652,-0.091656,0.312128,0.221963,-0.049410,0.394963,-0.066101,0.109899,0.072875,...,0.035922,0.245670,0.310494,-0.097445,-0.038705,-0.068324,-0.027750,-0.171403,-1.567976,2.696691
std,0.107995,0.070311,0.091085,0.130252,0.260921,0.037091,0.111554,0.044227,0.169603,0.072104,...,0.045836,0.076481,0.064397,0.149072,0.061731,0.055843,0.054115,0.165558,0.234706,2.662503
min,-0.414492,-0.170474,-0.428855,-0.189030,-0.202179,-0.231486,-0.068627,-0.311310,-0.246488,-0.289764,...,-0.243446,-0.108656,-0.095644,-0.376422,-0.333230,-0.330913,-0.217874,-0.518213,-1.966708,0.000000
25%,-0.281451,0.188339,-0.169851,0.190794,-0.047185,-0.071939,0.291742,-0.092717,-0.059577,0.012137,...,0.006500,0.183361,0.254403,-0.253409,-0.074627,-0.115565,-0.069625,-0.334869,-1.811119,1.000000
50%,-0.158354,0.232108,-0.103474,0.328118,0.260320,-0.047380,0.402840,-0.067295,0.129721,0.075557,...,0.037762,0.241674,0.312618,-0.076764,-0.044577,-0.072631,-0.030407,-0.139756,-1.547125,1.000000
75%,-0.075722,0.270721,-0.017998,0.433676,0.481040,-0.026329,0.496704,-0.042365,0.272931,0.133934,...,0.065301,0.317796,0.364526,0.041555,-0.002921,-0.017938,0.012859,-0.010328,-1.341360,5.000000
max,0.226545,0.548050,0.216811,0.691828,0.913784,0.229549,0.696488,0.149715,0.717532,0.328994,...,0.212165,0.430578,0.599509,0.401128,0.167531,0.099039,0.294926,0.475211,-0.598945,9.000000


In [6]:
# show the value count of the rid
genslm_embed['rid'].value_counts()

,count
rid,
1,7210
7,1326
6,694
3,519
8,496
0,405
5,297
2,284
9,282


In [7]:
# split the data into feature and target
X = genslm_embed.iloc[:, :-1].values
y = genslm_embed.iloc[:, -1].values

# split the data into training and test set with stratify
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

In [8]:
# scale the data
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Task 1: Multi-Class Classification (20 points)

Apply multiple classifiers to the dataset and identify the best-performing model based on the **macro F1 score**.  
The classifiers to evaluate include: **Multinomial Logistic Regression, LightGBM, XGBoost, and CatBoost**.

Please display the performance metrics for each classifier (e.g., **accuracy, macro precision, macro recall, macro F1**) to justify your selection of the top model for the downstream tasks.

In [9]:
# Set PyTorch as the backend for Keras before importing Keras
import os
os.environ["KERAS_BACKEND"] = "torch"

# Now import Keras (it will use PyTorch as backend)
import keras
from keras.models import Sequential
from keras.layers import Dense

# build a neural network classifier on the data
keras_classifier = Sequential(
    [
        Dense(25, activation = 'relu'),
        Dense(15, activation = 'relu'),
        Dense(10, activation = 'softmax')    # < softmax activation here
    ]
)

keras_classifier.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(),
    optimizer=keras.optimizers.Adam(0.001),
)

keras_classifier.fit(
    X_train, y_train,
    epochs=40
)


Epoch 1/40
294/294 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 1.0792
Epoch 2/40
294/294 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.4863
Epoch 3/40
294/294 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.2839
Epoch 4/40
294/294 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.2357
Epoch 5/40
294/294 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.1894
Epoch 6/40
294/294 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.1757
Epoch 7/40
294/294 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.1803
Epoch 8/40
294/294 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.1491
Epoch 9/40
294/294 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.1559
Epoch 10/40
294/294 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.1380
Epoch 11/40
294/294 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.1296
Epoch 12/40
294/294 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.1259
Epoch 13/40
294/294 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.1154
Epoch 14/40
294/294 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.1047
Epoch 15/40
294/294 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - lo

In [10]:
# show first few predicted probability vectors from the Keras (torch-backend) model
p_nonpreferred = keras_classifier.predict(X_test)

74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [11]:

y_pred = np.argmax(p_nonpreferred, axis=1)

## output the performance metrics
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

## output the precision, recall and f1 score
from sklearn.metrics import precision_score, recall_score, f1_score

print("Precision Multinomial classification: {:.2f}".format(precision_score(y_test, y_pred, average='macro')))
print("Recall Multinomial classification: {:.2f}".format(recall_score(y_test, y_pred, average='macro')))
print("F1 Score Multinomial classification: {:.2f}".format(f1_score(y_test, y_pred, average='macro')))

              precision    recall  f1-score   support

           0       0.95      0.90      0.92        81
           1       1.00      0.99      0.99      1442
           2       0.70      0.91      0.79        57
           3       0.74      0.71      0.73       104
           4       0.95      0.93      0.94        42
           5       0.86      0.80      0.83        60
           6       0.99      0.99      0.99       139
           7       0.94      0.92      0.93       265
           8       0.74      0.79      0.76        99
           9       0.88      0.89      0.88        56

    accuracy                           0.95      2345
   macro avg       0.87      0.88      0.88      2345
weighted avg       0.95      0.95      0.95      2345

Precision Multinomial classification: 0.87
Recall Multinomial classification: 0.88
F1 Score Multinomial classification: 0.88


In [12]:
#Now lets use LightGBM
import lightgbm as lgb
lgbm = lgb.LGBMClassifier(objective='multiclass',
                          num_class=10)
lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_test)
print("LightGBM Classification Report:")
print(classification_report(y_test, y_pred_lgbm))


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.909248 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 979200
[LightGBM] [Info] Number of data points in the train set: 9379, number of used features: 3840
[LightGBM] [Info] Start training from score -3.365485
[LightGBM] [Info] Start training from score -0.486148
[LightGBM] [Info] Start training from score -3.721278
[LightGBM] [Info] Start training from score -3.117950
[LightGBM] [Info] Start training from score -4.016330
[LightGBM] [Info] Start training from score -3.678168
[LightGBM] [Info] Start training from score -2.827260
[LightGBM] [Info] Start training from score -2.179261
[LightGBM] [Info] Start training from score -3.162292
[LightGBM] [Info] Start training from score -3.725693
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warnin

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.88      0.93        81
           1       0.98      1.00      0.99      1442
           2       0.87      0.68      0.76        57
           3       0.71      0.68      0.70       104
           4       1.00      0.88      0.94        42
           5       0.80      0.80      0.80        60
           6       0.99      0.97      0.98       139
           7       0.89      0.95      0.92       265
           8       0.73      0.66      0.69        99
           9       0.91      0.91      0.91        56

    accuracy                           0.94      2345
   macro avg       0.89      0.84      0.86      2345
weighted avg       0.94      0.94      0.94      2345



In [13]:

#Results from LGBM

print(classification_report(y_test, y_pred_lgbm))

## output the precision, recall and f1 score
from sklearn.metrics import precision_score, recall_score, f1_score

print("Precision: {:.2f}".format(precision_score(y_test, y_pred_lgbm, average='macro')))
print("Recall: {:.2f}".format(recall_score(y_test, y_pred_lgbm, average='macro')))
print("F1 Score: {:.2f}".format(f1_score(y_test, y_pred_lgbm, average='macro')))


              precision    recall  f1-score   support

           0       1.00      0.88      0.93        81
           1       0.98      1.00      0.99      1442
           2       0.87      0.68      0.76        57
           3       0.71      0.68      0.70       104
           4       1.00      0.88      0.94        42
           5       0.80      0.80      0.80        60
           6       0.99      0.97      0.98       139
           7       0.89      0.95      0.92       265
           8       0.73      0.66      0.69        99
           9       0.91      0.91      0.91        56

    accuracy                           0.94      2345
   macro avg       0.89      0.84      0.86      2345
weighted avg       0.94      0.94      0.94      2345

Precision: 0.89
Recall: 0.84
F1 Score: 0.86


In [15]:
#Use xgboost
import xgboost as xgb
xgb_model = xgb.XGBClassifier(objective='multi:softmax',
                              num_class=10)


xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

In [16]:
#Results from XGBOOST

print(classification_report(y_test, y_pred_xgb))

## output the precision, recall and f1 score
from sklearn.metrics import precision_score, recall_score, f1_score

print("Precision: {:.2f}".format(precision_score(y_test, y_pred_xgb, average='macro')))
print("Recall: {:.2f}".format(recall_score(y_test, y_pred_xgb, average='macro')))
print("F1 Score: {:.2f}".format(f1_score(y_test, y_pred_xgb, average='macro')))

              precision    recall  f1-score   support

           0       0.99      0.88      0.93        81
           1       0.98      1.00      0.99      1442
           2       0.82      0.74      0.78        57
           3       0.76      0.71      0.74       104
           4       1.00      0.88      0.94        42
           5       0.77      0.77      0.77        60
           6       0.99      0.96      0.97       139
           7       0.90      0.95      0.92       265
           8       0.78      0.73      0.75        99
           9       0.92      0.86      0.89        56

    accuracy                           0.94      2345
   macro avg       0.89      0.85      0.87      2345
weighted avg       0.94      0.94      0.94      2345

Precision: 0.89
Recall: 0.85
F1 Score: 0.87


In [18]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.2 MB/s eta 0:00:00


In [20]:
#Catboost for multiclass classification

from catboost import CatBoostClassifier
catboost_model = CatBoostClassifier(loss_function='MultiClass',
                                    task_type = "GPU")
catboost_model.fit(X_train, y_train, verbose = True)
y_pred_catboost = catboost_model.predict(X_test)

Learning rate set to 0.102472
0:	learn: 1.6991639	total: 296ms	remaining: 4m 55s
1:	learn: 1.4354643	total: 469ms	remaining: 3m 54s
2:	learn: 1.2561588	total: 636ms	remaining: 3m 31s
3:	learn: 1.1198878	total: 804ms	remaining: 3m 20s
4:	learn: 1.0108836	total: 970ms	remaining: 3m 12s
5:	learn: 0.9221921	total: 1.13s	remaining: 3m 7s
6:	learn: 0.8506827	total: 1.31s	remaining: 3m 5s
7:	learn: 0.7907017	total: 1.47s	remaining: 3m 2s
8:	learn: 0.7368114	total: 1.64s	remaining: 3m 1s
9:	learn: 0.6918476	total: 1.82s	remaining: 2m 59s
10:	learn: 0.6522343	total: 1.99s	remaining: 2m 58s
11:	learn: 0.6169180	total: 2.16s	remaining: 2m 58s
12:	learn: 0.5856078	total: 2.34s	remaining: 2m 57s
13:	learn: 0.5556048	total: 2.52s	remaining: 2m 57s
14:	learn: 0.5302127	total: 2.7s	remaining: 2m 57s
15:	learn: 0.5072179	total: 2.87s	remaining: 2m 56s
16:	learn: 0.4851333	total: 3.04s	remaining: 2m 56s
17:	learn: 0.4651458	total: 3.21s	remaining: 2m 55s
18:	learn: 0.4463506	total: 3.38s	remaining: 2m 5

In [21]:
#Results from CatBoost

print(classification_report(y_test, y_pred_catboost))

## output the precision, recall and f1 score
from sklearn.metrics import precision_score, recall_score, f1_score

print("Precision: {:.2f}".format(precision_score(y_test, y_pred_catboost, average='macro')))
print("Recall: {:.2f}".format(recall_score(y_test, y_pred_catboost, average='macro')))
print("F1 Score: {:.2f}".format(f1_score(y_test, y_pred_catboost, average='macro')))

              precision    recall  f1-score   support

           0       0.96      0.91      0.94        81
           1       0.98      1.00      0.99      1442
           2       0.84      0.74      0.79        57
           3       0.78      0.73      0.75       104
           4       1.00      0.93      0.96        42
           5       0.83      0.80      0.81        60
           6       1.00      0.98      0.99       139
           7       0.91      0.96      0.93       265
           8       0.81      0.74      0.77        99
           9       0.94      0.91      0.93        56

    accuracy                           0.95      2345
   macro avg       0.91      0.87      0.89      2345
weighted avg       0.95      0.95      0.95      2345

Precision: 0.91
Recall: 0.87
F1 Score: 0.89


Looking at the performance of all models we can conclude that the CatBoost model perfomed the best. It had the highest F1 score of 0.89, and precision score of 0.91. In addition, it had similar Recall and Accuracy to all other models.

## Task 2: Dimensionality Reduction for Efficient Classification (20 points)

Using the full embedding dimensionality for classification can be computationally intensive.  
In this step, your objective is to reduce the dimensionality of the data using **PCA** and **UMAP** to speed up classification.

Explore multiple values of `n_components` to evaluate how dimensionality affects classification performance.  
Use the **best-performing classifier identified in Task 1**.

The dimensionalities to explore are:

```python
n_components = [10, 20, 43, 100, 150, 200, 300, 400]
```

For each setting, report the **accuracy, macro precision, macro recall, and macro F1 score**.

After completing your experiments:

- **Identify which dimensionality reduction method (PCA or UMAP) performs better overall on this dataset.**
- **Determine the best-performing `n_components` value based on the evaluation metrics.**

**Note:** t-SNE is mainly used for reducing data to 2 or 3 dimensions for visualization.  
Because it does not scale effectively to higher-dimensional embeddings, we **exclude it** from this task.


## Task 3: Visualizing High-Dimensional Embeddings Using PCA, t-SNE, and UMAP (20 points)

Visualize the high-dimensional data using **PCA**, **t-SNE**, and **UMAP**.  
Keep in mind that tuning the hyperparameters for t-SNE and UMAP is essential for producing informative plots.

Experiment with different settings to generate the most insightful visualizations, and **explicitly report the best hyperparameters you identified through your experiments** in your submission.


## Task 4: Evaluating Clustering Algorithms on 2D Embeddings (30 points)

Clustering algorithms often struggle with the curse of dimensionality, which can lead to poor performance on high-dimensional data.  
To address this, use the **2-dimensional representation** obtained from **t-SNE** in Task 2 (`n_components = 2`, `perplexity = 25`).

Apply **three different clustering algorithms** to the 2D data and evaluate their effectiveness.  
For each algorithm, report the following:

- **Visualization** of the clustering result (dimension = 2)
- **Number of clusters**
- **Number of outliers** (`cluster_labels == -1`, if applicable)
- **Completeness score** and **Homogeneity score**  
  (refer to scikit-learn documentation for how to compute these metrics given ground-truth labels)

**Note:**  
Since clustering is unsupervised, use the entire dataset when tuning hyperparameters (no train-test split needed).


## Task 5: Summary of Best Results and Reflection (10 points)

Summarize the best results you obtained from each of the previous tasks and explain what you learned from this assignment.


# Task
Train a LightGBM Classifier model on `X_train` and `y_train`, predict on `X_test`, and then compute and print the accuracy, macro precision, macro recall, and macro F1 score of the predictions.

## Implement LightGBM Classifier

### Subtask:
Train a LightGBM classifier model on the training data, make predictions on the test data, and then evaluate its performance using accuracy, macro precision, macro recall, and macro F1 score.


**Reasoning**:
Implement the LightGBM classifier as requested, including importing necessary libraries, initializing and training the model, making predictions, and evaluating its performance using accuracy, macro precision, macro recall, and macro F1 score.



In [8]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Initialize LGBMClassifier
# Using objective='multiclass' and num_class=10 as there are 10 genome IDs
lgbm_model = lgb.LGBMClassifier(objective='multiclass', num_class=10, random_state=42)

# Train the model
print("Training LightGBM model...")
lgbm_model.fit(X_train, y_train)
print("LightGBM model training complete.")

# Make predictions on the test set
y_pred_lgbm = lgbm_model.predict(X_test)

# Calculate evaluation metrics
accuracy_lgbm = accuracy_score(y_test, y_pred_lgbm)
precision_lgbm = precision_score(y_test, y_pred_lgbm, average='macro', zero_division=0)
recall_lgbm = recall_score(y_test, y_pred_lgbm, average='macro', zero_division=0)
f1_lgbm = f1_score(y_test, y_pred_lgbm, average='macro', zero_division=0)

# Print the results
print(f"\nLightGBM Classifier Performance:")
print(f"Accuracy: {accuracy_lgbm:.4f}")
print(f"Macro Precision: {precision_lgbm:.4f}")
print(f"Macro Recall: {recall_lgbm:.4f}")
print(f"Macro F1-Score: {f1_lgbm:.4f}")

Training LightGBM model...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.178876 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 979200
[LightGBM] [Info] Number of data points in the train set: 9379, number of used features: 3840
[LightGBM] [Info] Start training from score -3.365485
[LightGBM] [Info] Start training from score -0.486148
[LightGBM] [Info] Start training from score -3.721278
[LightGBM] [Info] Start training from score -3.117950
[LightGBM] [Info] Start training from score -4.016330
[LightGBM] [Info] Start training from score -3.678168
[LightGBM] [Info] Start training from score -2.827260
[LightGBM] [Info] Start training from score -2.179261
[LightGBM] [Info] Start training from score -3.162292
[LightGBM] [Info] Start training from score -3.725693
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

KeyboardInterrupt: 